# Example queries: `restrict_avoid` (resstock_oedi)

Auto-generated from `tests/query_snapshots/restrict_avoid.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `restrict_single_state`

Annual electricity restricted to CO. Three equivalent restrict shapes (single-element list, scalar string, scalar wrapped in tuple) must all compile to the same SQL — covers the arg-normalization code path in _get_restrict_clauses.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 4)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption
                  Mobile Home           391 9.864994e+04                          7.898215e+08
Multi-Family with 2 - 4 Units           469 1.183295e+05                          8.547447e+08
   Multi-Family with 5+ Units          2001 5.048556e+05                          3.249347e+09
       Single-Family Attached           664 1.675283e+05                          1.345615e+09
       Single-Family Detached          5900 1.488580e+06                          1.635437e+10


## `restrict_two_states`

Annual electricity restricted to CO + WY.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['state'],
    restrict=[('state', ['CO', 'WY'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (2, 4)
state  sample_count  units_count  electricity.total.energy_consumption
   CO          9425 2.377943e+06                          2.259390e+10
   WY          1105 2.787933e+05                          2.886849e+09


## `restrict_state_plus_vintage`

CO + specific vintage bucket, electricity.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['vintage'],
    restrict=[('state', ['CO']), ('vintage', ['1980s'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (1, 4)
vintage  sample_count   units_count  electricity.total.energy_consumption
  1980s          1362 343634.831951                          3.374342e+09


## `avoid_building_type`

CO only, avoid one building type per schema.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    avoid=[('geometry_building_type_recs', ['Mobile Home'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (4, 4)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption
Multi-Family with 2 - 4 Units           469 1.183295e+05                          8.547447e+08
   Multi-Family with 5+ Units          2001 5.048556e+05                          3.249347e+09
       Single-Family Attached           664 1.675283e+05                          1.345615e+09
       Single-Family Detached          5900 1.488580e+06                          1.635437e+10


## `avoid_building_type_multi`

CO only, avoid multiple building types via multi-element avoid list — exercises the NOT IN code path in _add_avoid (single-value avoid emits != instead).


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    avoid=[('geometry_building_type_recs', ['Mobile Home', 'Multi-Family with 5+ Units'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (3, 4)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption
Multi-Family with 2 - 4 Units           469 1.183295e+05                          8.547447e+08
       Single-Family Attached           664 1.675283e+05                          1.345615e+09
       Single-Family Detached          5900 1.488580e+06                          1.635437e+10
